# Kaggle house prices - attempt

the ames housing dataset. target: low rmse on the public leaderboard.

plan: lots of feature engineering on a small dataset, stacked models.


In [ ]:
import pandas as pd
import numpy as np

tr = pd.read_csv('data/house-prices/train.csv')
te = pd.read_csv('data/house-prices/test.csv')
print(tr.shape, te.shape)
print(tr['SalePrice'].describe())


In [ ]:
# log target since prices are skewed
tr['logSalePrice'] = np.log1p(tr['SalePrice'])
tr['logSalePrice'].hist(bins=50)


In [ ]:
# missing value heatmap
import seaborn as sns
missing = tr.isnull().sum().sort_values(ascending=False)
missing[missing > 0]


In [ ]:
# fill numeric with median, categorical with 'None'
num = tr.select_dtypes(include=[np.number]).columns
cat = tr.select_dtypes(include=['object']).columns
for c in num:
    tr[c] = tr[c].fillna(tr[c].median())
    te[c] = te[c].fillna(tr[c].median()) if c in te else None
for c in cat:
    tr[c] = tr[c].fillna('None')
    te[c] = te[c].fillna('None')


In [ ]:
# ordinal encoding for ordered cats
quality_map = {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'None':0}
for c in ['ExterQual','ExterCond','BsmtQual','HeatingQC','KitchenQual']:
    tr[c] = tr[c].map(quality_map).fillna(0)
    te[c] = te[c].map(quality_map).fillna(0)


In [ ]:
# new features
for df in [tr, te]:
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['Age'] = df['YrSold'] - df['YearBuilt']
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)


In [ ]:
# dummies for the rest
combined = pd.concat([tr.drop(columns='SalePrice'), te])
combined = pd.get_dummies(combined)
tr_X = combined.iloc[:len(tr)]
te_X = combined.iloc[len(tr):]
print(tr_X.shape, te_X.shape)


In [ ]:
# baseline ridge
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
rid = Ridge(alpha=10)
y = tr['logSalePrice']
scores = -cross_val_score(rid, tr_X.drop(columns=['logSalePrice'], errors='ignore'), y, scoring='neg_root_mean_squared_error', cv=5)
print(scores.mean(), scores.std())


In [ ]:
# lasso for feature selection
from sklearn.linear_model import Lasso
las = Lasso(alpha=0.0005, max_iter=20000)
scores = -cross_val_score(las, tr_X.drop(columns=['logSalePrice'], errors='ignore'), y, scoring='neg_root_mean_squared_error', cv=5)
print(scores.mean(), scores.std())


In [ ]:
# xgboost
import xgboost as xgb
gb = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=4)
scores = -cross_val_score(gb, tr_X.drop(columns=['logSalePrice'], errors='ignore'), y, scoring='neg_root_mean_squared_error', cv=5)
print(scores.mean())


### combining ridge + xgb gives the best public lb so far.


In [ ]:
# stacked: avg of ridge + xgb predictions
rid.fit(tr_X.drop(columns=['logSalePrice'], errors='ignore'), y)
gb.fit(tr_X.drop(columns=['logSalePrice'], errors='ignore'), y)
p1 = rid.predict(te_X)
p2 = gb.predict(te_X)
pred = np.expm1(0.5*p1 + 0.5*p2)


In [ ]:
sub = pd.DataFrame({'Id': te['Id'], 'SalePrice': pred})
sub.to_csv('submission.csv', index=False)
sub.head()


### public leaderboard rmse: 0.119. landed in top-30%.


exp note: lr=3e-4 was the sweet spot.

In [ ]:
# saved

In [ ]:
# saved

In [ ]:
# checkpoint
# results above were the best so far

In [ ]:
import torch
torch.manual_seed(0)


In [ ]:
# add weight decay
# opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)


In [ ]:
import torch
torch.manual_seed(0)


swapped to mlflow autolog. fewer lines, same artifacts.

todo: add a real eval split. holdout was leaking validation.

todo: add a real eval split. holdout was leaking validation.

In [ ]:
for i in range(3):
    print(i)